# 06. WOE Binning and IV Analysis

## Objective

This notebook performs Weight of Evidence (WOE) binning and Information Value (IV) analysis for all candidate model features created in Notebook 05.

Main outputs:

- WOE/IV table for every candidate variable
- IV ranking
- WOE-transformed Train / Validation / OOT datasets
- correlation matrix on WOE-transformed variables
- diagnostic charts for each variable
- Excel and pickle artifacts

The notebook reads from:

`../data/processed/05.feature_engineering/`

and saves to:

`../data/processed/06.woe_binning_iv_analysis/`


In [ ]:
import os
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

## 1. Paths and input files

In [ ]:
PROJECT_ROOT = Path("..")

INPUT_DIR = PROJECT_ROOT / "data" / "processed" / "05.feature_engineering"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "06.woe_binning_iv_analysis"
CHARTS_DIR = OUTPUT_DIR / "charts"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = INPUT_DIR / "train_fe.pkl"
VALIDATION_FILE = INPUT_DIR / "validation_fe.pkl"
OOT_FILE = INPUT_DIR / "oot_test_fe.pkl"
FEATURES_FILE = INPUT_DIR / "candidate_model_features.pkl"

for path in [TRAIN_FILE, VALIDATION_FILE, OOT_FILE, FEATURES_FILE]:
    if not path.exists():
        raise FileNotFoundError(f"Missing expected input file: {path}")

print(f"Input directory: {INPUT_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Charts directory: {CHARTS_DIR.resolve()}")

## 2. Load datasets and candidate feature list

In [ ]:
train = pd.read_pickle(TRAIN_FILE)
validation = pd.read_pickle(VALIDATION_FILE)
oot_test = pd.read_pickle(OOT_FILE)

with open(FEATURES_FILE, "rb") as f:
    candidate_features = pickle.load(f)

TARGET_COL = "target_bad"

for dataset_name, data in {"train": train, "validation": validation, "oot_test": oot_test}.items():
    if TARGET_COL not in data.columns:
        raise ValueError(f"{TARGET_COL} missing from {dataset_name}.")

candidate_features = [c for c in candidate_features if c in train.columns and c != TARGET_COL]

print(f"Train shape:      {train.shape}")
print(f"Validation shape: {validation.shape}")
print(f"OOT shape:        {oot_test.shape}")
print(f"Candidate features: {len(candidate_features)}")

## 3. Initial feature screening

Before WOE/IV binning we exclude variables that are not suitable for automatic binning:

- missing rate above 70%
- constant variables
- near-constant variables with top category share above 99.5%
- very high-cardinality categorical variables above 100 unique levels


In [ ]:
MAX_MISSING_PCT = 0.70
MAX_TOP_SHARE = 0.995
MAX_CATEGORICAL_LEVELS = 100

screening_rows = []

for col in candidate_features:
    s = train[col]
    missing_pct = s.isna().mean()
    nunique = s.nunique(dropna=True)
    top_share = s.value_counts(dropna=False, normalize=True).iloc[0] if len(s) > 0 else np.nan

    is_numeric = pd.api.types.is_numeric_dtype(s)
    dtype_group = "numeric" if is_numeric else "categorical"

    exclusion_reason = None

    if missing_pct > MAX_MISSING_PCT:
        exclusion_reason = "missing_pct_above_70pct"
    elif nunique <= 1:
        exclusion_reason = "constant_or_single_unique_value"
    elif top_share > MAX_TOP_SHARE:
        exclusion_reason = "near_constant_top_share_above_99_5pct"
    elif (not is_numeric) and nunique > MAX_CATEGORICAL_LEVELS:
        exclusion_reason = "categorical_too_many_levels"

    screening_rows.append({
        "variable": col,
        "dtype": str(s.dtype),
        "dtype_group": dtype_group,
        "missing_pct_train": missing_pct,
        "nunique_train": nunique,
        "top_share_train": top_share,
        "use_for_woe": exclusion_reason is None,
        "exclusion_reason": exclusion_reason,
    })

feature_screening = pd.DataFrame(screening_rows)

woe_features = feature_screening.loc[
    feature_screening["use_for_woe"], "variable"
].tolist()

print(f"Features before screening: {len(candidate_features)}")
print(f"Features for WOE/IV:       {len(woe_features)}")
print(f"Excluded features:         {len(candidate_features) - len(woe_features)}")

feature_screening.sort_values(["use_for_woe", "missing_pct_train"], ascending=[True, False]).head(50)

## 4. Helper functions

The binning is fitted only on the training sample.


In [ ]:
def clean_sheet_name(name):
    cleaned = "".join(ch if ch.isalnum() or ch in ["_", "-"] else "_" for ch in str(name))
    return cleaned[:31]


def get_variable_dtype(data, variable):
    return "numerical" if pd.api.types.is_numeric_dtype(data[variable]) else "categorical"


def extract_iv_from_binning_table(binning_table_df):
    candidates = ["IV", "iv"]
    for c in candidates:
        if c in binning_table_df.columns:
            return pd.to_numeric(binning_table_df[c], errors="coerce").sum()
    return np.nan


def normalize_binning_table(table, variable):
    out = table.copy()
    out.insert(0, "variable", variable)

    rename_map = {
        "Count": "count",
        "Count (%)": "count_pct",
        "Non-event": "good_count",
        "Event": "bad_count",
        "Event rate": "bad_rate",
        "WoE": "woe",
        "IV": "iv",
        "JS": "js",
    }

    out = out.rename(columns={k: v for k, v in rename_map.items() if k in out.columns})
    return out


MISSING_NUMERIC_SENTINEL = -999999999
MISSING_CATEGORICAL_LABEL = "__MISSING__"


def prepare_binning_series(series, dtype_group):
    """Prepare a feature for OptimalBinning fit/transform.

    Some optbinning versions raise a generic `Missing value` error when
    raw NaN values are passed directly. We therefore map missing values to
    explicit placeholders consistently during both fit and transform.
    """
    if dtype_group == "numerical":
        return (
            pd.to_numeric(series, errors="coerce")
            .replace([np.inf, -np.inf], np.nan)
            .astype(float)
            .fillna(MISSING_NUMERIC_SENTINEL)
        )

    return (
        series.astype("object")
        .where(series.notna(), MISSING_CATEGORICAL_LABEL)
        .astype(str)
    )


def safe_transform(optb, series):
    dtype_group = getattr(optb, "dtype", None)
    if dtype_group not in ["numerical", "categorical"]:
        dtype_group = "numerical" if pd.api.types.is_numeric_dtype(series) else "categorical"

    prepared_series = prepare_binning_series(series, dtype_group)
    transformed = optb.transform(prepared_series, metric="woe")
    return pd.Series(transformed, index=series.index)


def is_bad_rate_monotonic(table):
    if "bad_rate" not in table.columns:
        return np.nan

    usable = table.copy()
    usable = usable[~usable["Bin"].astype(str).isin(["Special", "Missing", "Totals"])]
    rates = pd.to_numeric(usable["bad_rate"], errors="coerce").dropna()

    if len(rates) <= 2:
        return True

    return bool(rates.is_monotonic_increasing or rates.is_monotonic_decreasing)


def save_variable_chart(table, variable, charts_dir):
    usable = table.copy()
    if "Bin" not in usable.columns:
        return

    usable = usable[~usable["Bin"].astype(str).isin(["Totals"])]

    if "count" not in usable.columns or "bad_rate" not in usable.columns:
        return

    x = usable["Bin"].astype(str)
    counts = pd.to_numeric(usable["count"], errors="coerce")
    bad_rates = pd.to_numeric(usable["bad_rate"], errors="coerce")

    fig, ax1 = plt.subplots(figsize=(12, 5))

    ax1.bar(x, counts, alpha=0.75)
    ax1.set_ylabel("Count")
    ax1.set_xlabel(variable)
    ax1.tick_params(axis="x", rotation=45)
    ax1.grid(axis="y", linestyle="--", alpha=0.3)

    ax2 = ax1.twinx()
    ax2.plot(x, bad_rates, color="red", marker="o", linewidth=2.5)
    ax2.set_ylabel("Observed bad rate", color="red")
    ax2.tick_params(axis="y", labelcolor="red")

    plt.title(f"{variable}: WOE Binning Volume and Bad Rate")
    plt.tight_layout()

    file_name = "".join(ch if ch.isalnum() or ch in ["_", "-"] else "_" for ch in variable)
    plt.savefig(charts_dir / f"{file_name}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

## 5. Fit WOE binning for all variables

For numeric variables:

- `dtype="numerical"`
- `monotonic_trend="auto"`

For categorical variables:

- `dtype="categorical"`

Variables that fail automatic binning are logged and excluded from the WOE-transformed datasets.


In [ ]:
from optbinning import OptimalBinning

OPTBINNING_AVAILABLE = True

if not OPTBINNING_AVAILABLE:
    raise ImportError("optbinning is required for this notebook. Run: pip install optbinning")

y_train = train[TARGET_COL].astype(int)

binning_objects = {}
woe_tables = {}
iv_rows = []
failed_variables = []

for i, col in enumerate(woe_features, start=1):
    print(f"[{i}/{len(woe_features)}] Binning: {col}")

    dtype = get_variable_dtype(train, col)
    x = prepare_binning_series(train[col], dtype)

    try:
        if dtype == "numerical":
            optb = OptimalBinning(
                name=col,
                dtype="numerical",
                monotonic_trend="auto",
                solver="cp",
            )
        else:
            optb = OptimalBinning(
                name=col,
                dtype="categorical",
                solver="cp",
            )

        optb.fit(x, y_train)
        table = optb.binning_table.build()
        table_norm = normalize_binning_table(table, col)

        iv = extract_iv_from_binning_table(table)
        mono = is_bad_rate_monotonic(table_norm)

        binning_objects[col] = optb
        woe_tables[col] = table_norm

        iv_rows.append({
            "variable": col,
            "dtype_group": dtype,
            "iv": iv,
            "n_bins": table_norm[~table_norm["Bin"].astype(str).isin(["Totals"])].shape[0],
            "bad_rate_monotonic": mono,
            "status": "success",
            "error": None,
        })

        save_variable_chart(table_norm, col, CHARTS_DIR)

    except Exception as e:
        failed_variables.append({
            "variable": col,
            "dtype_group": dtype,
            "error": str(e),
        })
        iv_rows.append({
            "variable": col,
            "dtype_group": dtype,
            "iv": np.nan,
            "n_bins": np.nan,
            "bad_rate_monotonic": np.nan,
            "status": "failed",
            "error": str(e),
        })

iv_summary = pd.DataFrame(iv_rows).sort_values("iv", ascending=False)
failed_variables_df = pd.DataFrame(failed_variables)

print("Completed.")
print(f"Successful variables: {len(binning_objects)}")
print(f"Failed variables:     {len(failed_variables_df)}")

iv_summary.head(50)

## 6. IV interpretation and initial ranking

Common IV interpretation:

| IV range | Interpretation |
|---:|---|
| < 0.02 | very weak / not useful |
| 0.02 - 0.10 | weak |
| 0.10 - 0.30 | medium |
| 0.30 - 0.50 | strong |
| > 0.50 | very strong / investigate for leakage or redundancy |

Variables such as `grade`, `sub_grade`, and `int_rate` may contain lender pricing / underwriting information and should be reviewed carefully.


In [ ]:
def iv_band(iv):
    if pd.isna(iv):
        return "failed"
    if iv < 0.02:
        return "very_weak"
    if iv < 0.10:
        return "weak"
    if iv < 0.30:
        return "medium"
    if iv < 0.50:
        return "strong"
    return "very_strong_investigate"

iv_summary["iv_band"] = iv_summary["iv"].apply(iv_band)

iv_summary.head(100)

In [ ]:
# IV ranking chart
plot_iv = iv_summary[
    (iv_summary["status"] == "success") & iv_summary["iv"].notna()
].head(30).sort_values("iv", ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
ax.barh(plot_iv["variable"], plot_iv["iv"], alpha=0.8)
ax.set_xlabel("Information Value")
ax.set_ylabel("Variable")
ax.set_title("Top 30 Variables by IV")
ax.grid(axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig(CHARTS_DIR / "top_30_iv_ranking.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. WOE transformation

The fitted binning objects are used to transform Train, Validation and OOT into WOE values.

The target and audit fields are preserved.


In [ ]:
def transform_to_woe(data, binning_objects, target_col=TARGET_COL):
    out = pd.DataFrame(index=data.index)

    for col, optb in binning_objects.items():
        out[f"{col}_woe"] = safe_transform(optb, data[col])

    out[target_col] = data[target_col].values

    for audit_col in ["issue_d", "issue_quarter", "issue_month", "sample"]:
        if audit_col in data.columns:
            out[audit_col] = data[audit_col].values

    return out


train_woe = transform_to_woe(train, binning_objects)
validation_woe = transform_to_woe(validation, binning_objects)
oot_test_woe = transform_to_woe(oot_test, binning_objects)

print(train_woe.shape, validation_woe.shape, oot_test_woe.shape)
train_woe.head()

## 8. WOE missingness and sanity checks

In [ ]:
woe_feature_cols = [c for c in train_woe.columns if c.endswith("_woe")]

woe_missingness = pd.DataFrame({
    "variable": woe_feature_cols,
    "train_missing_pct": [train_woe[c].isna().mean() for c in woe_feature_cols],
    "validation_missing_pct": [validation_woe[c].isna().mean() for c in woe_feature_cols],
    "oot_missing_pct": [oot_test_woe[c].isna().mean() for c in woe_feature_cols],
})

woe_missingness["max_missing_pct"] = woe_missingness[
    ["train_missing_pct", "validation_missing_pct", "oot_missing_pct"]
].max(axis=1)

woe_missingness.sort_values("max_missing_pct", ascending=False).head(50)

## 9. Correlation matrix on WOE variables

This is used after IV ranking to identify redundant variables.

Typical examples in this dataset may include:

- `grade`, `sub_grade`, `int_rate`
- original and capped versions of the same numeric feature
- FICO low/high/midpoint variants

In [ ]:
corr_matrix = train_woe[woe_feature_cols].corr()

corr_pairs = (
    corr_matrix
    .where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)

corr_pairs.columns = ["variable_1", "variable_2", "correlation"]
corr_pairs["abs_correlation"] = corr_pairs["correlation"].abs()
corr_pairs = corr_pairs.sort_values("abs_correlation", ascending=False)

high_corr_pairs = corr_pairs[corr_pairs["abs_correlation"] >= 0.70].copy()

high_corr_pairs.head(100)

In [ ]:
# Correlation heatmap for top IV variables
top_vars = (
    iv_summary[(iv_summary["status"] == "success") & (iv_summary["iv"].notna())]
    .head(25)["variable"]
    .map(lambda x: f"{x}_woe")
    .tolist()
)

top_vars = [c for c in top_vars if c in train_woe.columns]

if len(top_vars) >= 2:
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(train_woe[top_vars].corr(), aspect="auto")
    ax.set_xticks(range(len(top_vars)))
    ax.set_yticks(range(len(top_vars)))
    ax.set_xticklabels(top_vars, rotation=90)
    ax.set_yticklabels(top_vars)
    ax.set_title("Correlation Heatmap - Top IV WOE Variables")
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / "correlation_heatmap_top_iv_woe.png", dpi=150, bbox_inches="tight")
    plt.show()

## 10. Initial candidate selection flag
The flag below is only a first-pass recommendation for review:

- successful binning
- IV >= 0.02
- no excessive WOE missingness
- bad-rate monotonicity where available

Correlation and business judgement are reviewed separately.


In [ ]:
iv_for_selection = iv_summary.copy()
iv_for_selection["woe_variable"] = iv_for_selection["variable"] + "_woe"

iv_for_selection = iv_for_selection.merge(
    woe_missingness,
    left_on="woe_variable",
    right_on="variable",
    how="left",
    suffixes=("", "_woe_missingness")
)

iv_for_selection["initial_selection_candidate"] = (
    (iv_for_selection["status"] == "success")
    & (iv_for_selection["iv"] >= 0.02)
    & (iv_for_selection["max_missing_pct"].fillna(0) <= 0.05)
)

initial_selected_features = iv_for_selection.loc[
    iv_for_selection["initial_selection_candidate"],
    "variable"
].tolist()

initial_selected_woe_features = [f"{c}_woe" for c in initial_selected_features]

print(f"Initial selected features before correlation/business review: {len(initial_selected_features)}")

iv_for_selection.sort_values("iv", ascending=False).head(100)

## 11. Save all outputs

Outputs are saved into:

`../data/processed/06.woe_binning_iv_analysis/`

Charts are saved into:

`../data/processed/06.woe_binning_iv_analysis/charts/`


In [ ]:
# Save WOE transformed datasets
train_woe.to_pickle(OUTPUT_DIR / "train_woe.pkl")
validation_woe.to_pickle(OUTPUT_DIR / "validation_woe.pkl")
oot_test_woe.to_pickle(OUTPUT_DIR / "oot_test_woe.pkl")

# Save binning objects and feature lists
with open(OUTPUT_DIR / "woe_binning_objects.pkl", "wb") as f:
    pickle.dump(binning_objects, f)

with open(OUTPUT_DIR / "woe_feature_columns.pkl", "wb") as f:
    pickle.dump(woe_feature_cols, f)

with open(OUTPUT_DIR / "initial_selected_features.pkl", "wb") as f:
    pickle.dump(initial_selected_features, f)

with open(OUTPUT_DIR / "initial_selected_woe_features.pkl", "wb") as f:
    pickle.dump(initial_selected_woe_features, f)

# Save full artifacts
artifacts = {
    "target_col": TARGET_COL,
    "candidate_features": candidate_features,
    "feature_screening": feature_screening,
    "woe_features": woe_features,
    "iv_summary": iv_summary,
    "iv_for_selection": iv_for_selection,
    "woe_tables": woe_tables,
    "failed_variables": failed_variables_df,
    "woe_missingness": woe_missingness,
    "corr_matrix": corr_matrix,
    "corr_pairs": corr_pairs,
    "high_corr_pairs": high_corr_pairs,
    "initial_selected_features": initial_selected_features,
    "initial_selected_woe_features": initial_selected_woe_features,
}

with open(OUTPUT_DIR / "woe_binning_iv_artifacts.pkl", "wb") as f:
    pickle.dump(artifacts, f)

# Save Excel report
excel_path = OUTPUT_DIR / "woe_binning_iv_report.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    feature_screening.to_excel(writer, sheet_name="feature_screening", index=False)
    iv_summary.to_excel(writer, sheet_name="iv_summary", index=False)
    iv_for_selection.to_excel(writer, sheet_name="iv_selection", index=False)
    failed_variables_df.to_excel(writer, sheet_name="failed_variables", index=False)
    woe_missingness.to_excel(writer, sheet_name="woe_missingness", index=False)
    high_corr_pairs.to_excel(writer, sheet_name="high_corr_pairs", index=False)
    corr_pairs.head(5000).to_excel(writer, sheet_name="corr_pairs_top5000", index=False)

    pd.DataFrame({"initial_selected_features": initial_selected_features}).to_excel(
        writer,
        sheet_name="initial_selected",
        index=False,
    )

    for variable, table in woe_tables.items():
        sheet_name = clean_sheet_name(variable)
        table.to_excel(writer, sheet_name=sheet_name, index=False)

# Separate correlation matrix file, because it can be wide
corr_matrix.to_excel(OUTPUT_DIR / "woe_correlation_matrix.xlsx")

print("Saved outputs:")
print(f"- {OUTPUT_DIR / 'train_woe.pkl'}")
print(f"- {OUTPUT_DIR / 'validation_woe.pkl'}")
print(f"- {OUTPUT_DIR / 'oot_test_woe.pkl'}")
print(f"- {OUTPUT_DIR / 'woe_binning_objects.pkl'}")
print(f"- {OUTPUT_DIR / 'woe_binning_iv_artifacts.pkl'}")
print(f"- {excel_path}")
print(f"- {OUTPUT_DIR / 'woe_correlation_matrix.xlsx'}")
print(f"- charts saved in {CHARTS_DIR}")

## 12. Conclusions


1. Review of the Excel outputs related to:
   - `iv_summary`
   - `iv_selection`
   - `high_corr_pairs`
   - individual WOE sheets for key variables
2. Identify variables with:
   - strong IV
   - logical risk direction
   - acceptable monotonicity
   - no obvious leakage
   - no excessive correlation
3. Finalize candidate variables for Notebook 07: Logistic Regression Scorecard


## 13. Business Review of High IV Variables

Several variables exhibit very high Information Value (IV).

Before proceeding to logistic regression modelling, a business review is performed to identify variables that may represent lender decisions, pricing outputs or embedded risk assessments.

Such variables may artificially inflate model performance while reducing model interpretability and regulatory defensibility.

In [ ]:
business_review = pd.DataFrame(
    [
        {
            "variable": "sub_grade",
            "decision": "exclude",
            "reason": "contains lender underwriting assessment; potential embedded risk score",
        },
        {
            "variable": "grade",
            "decision": "exclude",
            "reason": "contains lender underwriting assessment; highly correlated with sub_grade",
        },
        {
            "variable": "int_rate",
            "decision": "exclude",
            "reason": "pricing variable derived from risk assessment",
        },
        {
            "variable": "verification_status",
            "decision": "exclude",
            "reason": "counterintuitive business relationship; likely policy driven",
        },
    ]
)

business_review

### Business Interpretation

Although `grade`, `sub_grade` and `int_rate` exhibit extremely high IV values, these variables are excluded from the baseline scorecard.

They are considered lender-generated risk assessments and pricing outputs, rather than primary borrower risk characteristics.

The objective of this model is to estimate risk using underlying credit characteristics rather than reproducing existing lender decisions.

## 14. Correlation Reduction

Highly correlated predictors can introduce redundancy and reduce model stability.

For pairs with absolute correlation above 0.70, the variable with the lower Information Value (IV) is removed.

The logic is:

`corr > 0.70` → keep the variable with higher IV → remove the variable with lower IV.

In [ ]:
CORR_THRESHOLD = 0.70

iv_lookup = (
    iv_summary
    .set_index("variable")["iv"]
    .to_dict()
)

high_corr_pairs_filtered = high_corr_pairs[
    high_corr_pairs["abs_correlation"] > CORR_THRESHOLD
].copy()

to_remove_corr = set()

for _, row in high_corr_pairs_filtered.iterrows():

    v1 = row["variable_1"].replace("_woe", "")
    v2 = row["variable_2"].replace("_woe", "")

    if v1 in to_remove_corr or v2 in to_remove_corr:
        continue

    iv1 = iv_lookup.get(v1, -999)
    iv2 = iv_lookup.get(v2, -999)

    if iv1 >= iv2:
        to_remove_corr.add(v2)
    else:
        to_remove_corr.add(v1)

corr_elimination = pd.DataFrame(
    {"variable_removed": sorted(to_remove_corr)}
)

corr_elimination

## 15. Remove Engineered Duplicates

Several engineered variables contain the same business information in a different representation.

In [ ]:
manual_duplicate_removals = [
    "loan_amnt_capped",
    "annual_inc_capped",
    "dti_capped",
    "revol_util_capped",
    "credit_history_months",
    "credit_history_months_capped",
    "credit_history_years_capped",
    "term_months",
    "is_60m_term",
    "fico_range_high",
    "fico_range_low",
]

manual_duplicate_elimination = pd.DataFrame(
    {"variable_removed": manual_duplicate_removals}
)

manual_duplicate_elimination

## 16. IV Threshold Review

Variables with IV below 0.10 are reviewed individually.

In [ ]:
IV_THRESHOLD = 0.10

low_iv_features = (
    iv_summary
    .query("iv < @IV_THRESHOLD")
    ["variable"]
    .tolist()
)

low_iv_review = pd.DataFrame(
    {"low_iv_variable": low_iv_features}
)

low_iv_review

### Additional Review: `loan_amnt`

Although `loan_amnt` remains statistically relevant and passes the IV screening process (IV ≈ 0.04), an additional business review suggests that the variable should be excluded from the final scorecard candidate set.

**Rationale**

- `loan_amnt` exhibits a **counterintuitive positive coefficient** when included in the multivariate WOE Logistic Regression model.
- The resulting scorecard assigns **higher score points to larger loan amounts**, despite larger loans showing higher observed bad rates in the univariate WOE analysis.
- The variable contributes only **limited standalone predictive power** (IV ≈ 0.04).
- Stronger affordability-related variables already exist in the model candidate set:
  - `loan_to_income`
  - `dti`
  - `annual_inc`

These variables capture the borrower's repayment capacity more directly and provide a more intuitive risk interpretation.

**Decision**

```text
Exclude loan_amnt from the final Logistic Regression candidate set.
```

This decision improves model interpretability, business consistency and scorecard explainability while having minimal expected impact on predictive performance.


## 17. Final Candidate Variables for Logistic Regression

The final candidate list is created after applying:

1. business exclusions,
2. correlation-based removals,
3. engineered duplicate removals,
4. manual judgement for variables that are interpretable and suitable for a borrower-level scorecard.

This list is the modelling input for the logistic regr scorecard.

In [ ]:
business_exclusions = business_review.loc[
    business_review["decision"] == "exclude",
    "variable"
].tolist()

preferred_final_candidate_features = [
    "fico_mid",
    "loan_to_income",
    "dti",
    "revol_util",
    "annual_inc",
    "credit_history_years",
    "inq_last_6mths",
    "acc_open_past_24mths",
    "num_tl_op_past_12m",
    "mort_acc",
    "purpose",
    "home_ownership",
    "term",
]

available_woe_variables = set(woe_tables.keys())

final_candidate_features = [
    v
    for v in preferred_final_candidate_features
    if v in available_woe_variables
    and v not in business_exclusions
    and v not in to_remove_corr
    and v not in manual_duplicate_removals
]


# derive the final list from the initial selection flags.
if len(final_candidate_features) == 0:
    final_candidate_features = [
        v
        for v in initial_selected_features
        if v in available_woe_variables
        and v not in business_exclusions
        and v not in to_remove_corr
        and v not in manual_duplicate_removals
    ]

final_candidate_woe_features = [
    f"{v}_woe"
    for v in final_candidate_features
    if f"{v}_woe" in train_woe.columns
]

final_candidate_variables = pd.DataFrame(
    {"final_candidate_variable": final_candidate_features}
)

print(f"Final candidate raw variables: {len(final_candidate_features)}")
print(f"Final candidate WOE variables: {len(final_candidate_woe_features)}")

final_candidate_variables

## 18. Final Candidate Correlation Matrix

The final correlation matrix is calculated only for the selected WOE-transformed variables.

This is the final multicollinearity check before logistic regression.

In [ ]:
if len(final_candidate_woe_features) >= 2:
    final_candidate_correlation_matrix = (
        train_woe[final_candidate_woe_features]
        .corr()
    )
else:
    final_candidate_correlation_matrix = pd.DataFrame()

final_candidate_correlation_matrix.round(3)

## 19. Save Final Modelling Inputs

The final selected variables, final WOE variables, binning tables and correlation matrix are saved as separate outputs.

These files are used as controlled inputs for the logistic regression scorecard notebook.

In [ ]:
# Save final candidate feature lists
with open(OUTPUT_DIR / "final_candidate_features.pkl", "wb") as f:
    pickle.dump(final_candidate_features, f)

with open(OUTPUT_DIR / "final_candidate_woe_features.pkl", "wb") as f:
    pickle.dump(final_candidate_woe_features, f)

# Save review outputs
business_review.to_excel(
    OUTPUT_DIR / "business_review.xlsx",
    index=False,
)

corr_elimination.to_excel(
    OUTPUT_DIR / "correlation_elimination.xlsx",
    index=False,
)

manual_duplicate_elimination.to_excel(
    OUTPUT_DIR / "manual_duplicate_elimination.xlsx",
    index=False,
)

low_iv_review.to_excel(
    OUTPUT_DIR / "low_iv_review.xlsx",
    index=False,
)

final_candidate_variables.to_excel(
    OUTPUT_DIR / "final_candidate_variables.xlsx",
    index=False,
)

final_candidate_correlation_matrix.to_excel(
    OUTPUT_DIR / "final_candidate_correlation_matrix.xlsx",
)

# Save final candidate binning tables only
final_binning_tables_path = OUTPUT_DIR / "final_candidate_binning_tables.xlsx"

with pd.ExcelWriter(final_binning_tables_path, engine="openpyxl") as writer:

    for variable in final_candidate_features:

        if variable in woe_tables:

            sheet_name = clean_sheet_name(variable)

            woe_tables[variable].to_excel(
                writer,
                sheet_name=sheet_name,
                index=False,
            )

# Save a compact final review workbook
final_review_path = OUTPUT_DIR / "final_variable_selection_review.xlsx"

with pd.ExcelWriter(final_review_path, engine="openpyxl") as writer:
    final_candidate_variables.to_excel(writer, sheet_name="final_candidates", index=False)
    business_review.to_excel(writer, sheet_name="business_review", index=False)
    corr_elimination.to_excel(writer, sheet_name="corr_elimination", index=False)
    manual_duplicate_elimination.to_excel(writer, sheet_name="duplicate_removals", index=False)
    low_iv_review.to_excel(writer, sheet_name="low_iv_review", index=False)

    if not final_candidate_correlation_matrix.empty:
        final_candidate_correlation_matrix.to_excel(writer, sheet_name="final_corr_matrix")

# Update full artifacts object with final selection outputs
final_selection_artifacts = {
    "business_review": business_review,
    "business_exclusions": business_exclusions,
    "corr_threshold": CORR_THRESHOLD,
    "corr_elimination": corr_elimination,
    "manual_duplicate_removals": manual_duplicate_removals,
    "manual_duplicate_elimination": manual_duplicate_elimination,
    "iv_threshold": IV_THRESHOLD,
    "low_iv_review": low_iv_review,
    "final_candidate_features": final_candidate_features,
    "final_candidate_woe_features": final_candidate_woe_features,
    "final_candidate_correlation_matrix": final_candidate_correlation_matrix,
}

with open(OUTPUT_DIR / "final_variable_selection_artifacts.pkl", "wb") as f:
    pickle.dump(final_selection_artifacts, f)

print("Saved final modelling inputs:")
print(f"- {OUTPUT_DIR / 'final_candidate_features.pkl'}")
print(f"- {OUTPUT_DIR / 'final_candidate_woe_features.pkl'}")
print(f"- {OUTPUT_DIR / 'final_candidate_variables.xlsx'}")
print(f"- {OUTPUT_DIR / 'final_candidate_binning_tables.xlsx'}")
print(f"- {OUTPUT_DIR / 'final_candidate_correlation_matrix.xlsx'}")
print(f"- {OUTPUT_DIR / 'final_variable_selection_review.xlsx'}")
print(f"- {OUTPUT_DIR / 'final_variable_selection_artifacts.pkl'}")